# quack-track: Colab Setup
Run this notebook to set up and train on Google Colab.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive for persistent storage
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# Clone repo and enter directory
REPO = 'https://github.com/yara-hue/quack-track.git'
if not os.path.isdir('/content/quack-track'):
    !git clone {REPO}
%cd /content/quack-track
print(f'Working in: {os.getcwd()}')

In [ ]:
# Pull latest fixes
!git pull

In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install opencv-python numpy matplotlib pyyaml tqdm thop -q
print('Dependencies installed')

In [ ]:
# Create dataset symlinks
DRIVE_DATA = '/content/drive/MyDrive/quack-track-data'
os.makedirs('data', exist_ok=True)

links = [
    ('UAV123_10fps', f'{DRIVE_DATA}/UAV123_10fps'),
    ('UAV20L', f'{DRIVE_DATA}/UAV20L'),
    ('train2017', f'{DRIVE_DATA}/COCO/train2017'),
]
for name, target in links:
    link_path = f'data/{name}'
    if not os.path.islink(link_path):
        os.symlink(target, link_path)
        print(f'Linked data/{name} -> {target}')
    else:
        print(f'data/{name} already linked')

# Checkpoints in Drive (persistent)
os.makedirs(f'{DRIVE_DATA}/checkpoints', exist_ok=True)
if not os.path.islink('checkpoints'):
    os.symlink(f'{DRIVE_DATA}/checkpoints', 'checkpoints')
    print('Linked checkpoints')

In [ ]:
# Verify dataset
root = 'data/UAV123_10fps'
if os.path.isdir(root):
    !python scripts/diagnose_dataset.py {root}
else:
    print(f'{root} not found - place dataset in Drive')

In [ ]:
# Test GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    !nvidia-smi

In [ ]:
print('Setup complete!')
print('Start training: python scripts/train.py --config configs/default.yaml')